# 02 — Palisades Mini-Pipeline (tilesvc serving mirror)

Phase 4 of `docs/v4-retraining-gameplan.md`. Every cell loads the **same** module
`services/tilesvc/` uses in production, so this notebook is a faithful mirror of the
live serving path. Iterate on features / model / calibration here without redeploying.

**Acceptance:** running cells 1–14 with the *v3* checkpoint reproduces the bad live
heatmap; with *v4* it should push SW and raise CSI vs the observed perimeter.


## 1. Config — pick event, ref_time, checkpoint


In [ ]:
import sys, os, json, datetime as dt
from pathlib import Path
REPO = Path.cwd().resolve()
while REPO.name and not (REPO/'services'/'tilesvc').exists(): REPO = REPO.parent
sys.path.insert(0, str(REPO)); os.chdir(REPO)
print('repo:', REPO)

# Event presets mirror frontend MapComponent.jsx HISTORICAL_FIRES
EVENT = {'name':'Palisades','lat':34.078,'lon':-118.555,
         'date':'2025-01-07T18:30:00Z','ignition':True}
TSEQ, STEPS, STEP_HOURS = 6, 6, 24
MODEL_PATH = os.environ.get('MODEL_PATH',
    'models/convlstm_unet_v3_delta_Cd13_Cs15_H64_T6_nautilus.pt')
EVENT, MODEL_PATH

## 2. Tile geometry — canonical EPSG:5070 grid (shared with tilesvc)


In [ ]:
from services.tilesvc.grid import lonlat_to_tile, tile_affine, tile_bounds_lonlat, build_grid, SIZE, TILE_M
tile = lonlat_to_tile(EVENT['lon'], EVENT['lat'])
bounds = tile_bounds_lonlat(tile)
print('tile', tile, 'SIZE', SIZE, 'TILE_M', TILE_M)
print('lonlat bounds', bounds)

## 3. Static tensor — per-channel stats + 3x3 visual


In [ ]:
from services.tilesvc.static_catalog import load_static_tensor_for_model
# returns (stat[Cs,H,W], static_order, meta) — see static_catalog.py for exact signature
stat, static_order, stat_meta = load_static_tensor_for_model(EVENT['lat'], EVENT['lon'])
import numpy as np
for i,n in enumerate(static_order):
    a = np.asarray(stat[i]); print(f'{n:12s} min={a.min():.3f} mean={a.mean():.3f} max={a.max():.3f}')

In [ ]:
import matplotlib.pyplot as plt
n=len(static_order); cols=3; rows=(n+cols-1)//cols
fig,ax=plt.subplots(rows,cols,figsize=(9,3*rows))
for i,n_ in enumerate(static_order):
    a=ax.flat[i]; a.imshow(stat[i]); a.set_title(n_); a.axis('off')
plt.tight_layout()

## 4. Dynamic FIRMS rasterization — fire_t for each history frame


In [ ]:
from services.tilesvc.dynamic_builder import build_dynamic_for_tile
ref_time = dt.datetime.fromisoformat(EVENT['date'].replace('Z','+00:00'))
dyn, dyn_meta = build_dynamic_for_tile(EVENT['lat'], EVENT['lon'], Tseq=TSEQ,
                                       ref_time=ref_time, ignition=EVENT['ignition'])
# dyn: [T, Cd, H, W]; visualize fire_t history
fire_idx = 0
fig,ax=plt.subplots(1,TSEQ,figsize=(2.2*TSEQ,2.4))
for t in range(TSEQ): ax[t].imshow(dyn[t,fire_idx]); ax[t].set_title(f't-{TSEQ-1-t}'); ax[t].axis('off')

## 5. HRRR weather — quiver; sanity-check wind direction for the event


In [ ]:
from services.tilesvc.dynamic_builder import fetch_weather_grids
wx = fetch_weather_grids(EVENT['lat'], EVENT['lon'], ref_time=ref_time)
u,v = np.asarray(wx['u']), np.asarray(wx['v'])
print('mean u (east):', float(u.mean()), ' mean v (north):', float(v.mean()),
      '-> Santa-Ana' if u.mean()<-5 else '-> not Santa-Ana')
step=max(1,SIZE//16)
yy,xx=np.mgrid[0:SIZE:step,0:SIZE:step]
plt.figure(figsize=(4,4)); plt.quiver(xx,yy,u[::step,::step],-v[::step,::step]); plt.gca().invert_yaxis(); plt.title('wind (toward)')

## 6. Derived features — verify wind_dir_cos/sin match the quiver


In [ ]:
from ignis_ml.src.data.features import append_derived_features
# dyn here is already normalized by the builder; pass its channel order
dyn_order = dyn_meta.get('dyn_order') if isinstance(dyn_meta,dict) else None
x_dyn_d, new_order = append_derived_features(np.asarray(dyn), dyn_order=dyn_order or [], days_since_fire_cap=7)
print('channels:', new_order)

## 7. Model inference — load checkpoint, one forward pass


In [ ]:
import torch
from services.tilesvc.ml_runtime import load_model_once  # mirrors app._load_model_once if exposed
# If ml_runtime doesn't expose a loader, replicate app.py::_load_model_once here.
model = load_model_once(MODEL_PATH)  # -> eval() ConvLSTMUNet
with torch.no_grad():
    xd = torch.from_numpy(np.asarray(x_dyn_d)[None]).float()
    xs = torch.from_numpy(np.asarray(stat)[None]).float()
    logits = model(xd, xs)
    prob = torch.sigmoid(logits)[0,0].cpu().numpy()
print('prob', prob.min(), prob.mean(), prob.max())
plt.imshow(prob); plt.colorbar(); plt.title('step-1 prob')

## 8. Multistep rollout — replicate app._rollout_multistep_predictions


In [ ]:
# Import the real rollout so the notebook and prod share code, not just behavior.
from services.tilesvc.app import _rollout_multistep_predictions
bounds5070, crop, rollout = _rollout_multistep_predictions(
    EVENT['lat'], EVENT['lon'], Tseq=TSEQ, steps=STEPS, step_hours=STEP_HOURS,
    crop_frac=1.0, ignition=EVENT['ignition'], ref_time=ref_time, threshold=0.5)
frames=[r['prob'] for r in rollout]
fig,ax=plt.subplots(1,len(frames),figsize=(2.2*len(frames),2.4))
for i,p in enumerate(frames): ax[i].imshow(p); ax[i].set_title(f'day {i+1}'); ax[i].axis('off')

## 9. Calibration — raw vs calibrated


In [ ]:
from services.tilesvc.calibration import load_calibration, apply_calibration  # see calibration.py
cal = load_calibration()
cal_frames=[apply_calibration(p, cal) for p in frames]  # adjust to actual fn name
print('calibration method:', cal.get('method'))

## 10. Observed ground truth — rasterize WFIGS/FRAP perimeter to tile grid


In [ ]:
from ignis_ml.scripts.eval_historical import rasterize_perimeter, Preset
preset = Preset('palisades', EVENT['name'], EVENT['lat'], EVENT['lon'], EVENT['date'])
obs = rasterize_perimeter(Path('data/perimeters/palisades.geojson'), preset, day=None)
plt.imshow(obs) if obs is not None else print('add data/perimeters/palisades.geojson')

## 11. Metrics — per-step IoU / Dice / CSI / Hausdorff (direction-aware)


In [ ]:
from ignis_ml.scripts.eval_historical import iou, dice, csi, hausdorff_km
thr=0.10
if obs is not None:
    for i,p in enumerate(frames,1):
        pred=(p>=thr).astype('float32')
        print(f'day {i}: IoU={iou(pred,obs):.3f} Dice={dice(pred,obs):.3f} CSI={csi(pred,obs):.3f} Hd={hausdorff_km(pred,obs):.2f}km')

## 12. Visualization — basemap + heatmap + observed outline + wind quiver
This is the headline figure to re-run after every retrain.


In [ ]:
fig,ax=plt.subplots(1,len(frames),figsize=(3*len(frames),3))
for i,p in enumerate(frames):
    ax[i].imshow(p, cmap='inferno', vmin=0, vmax=1)
    if obs is not None: ax[i].contour(obs, levels=[0.5], colors='cyan', linewidths=1)
    ax[i].quiver(xx,yy,u[::step,::step],-v[::step,::step], color='white', alpha=.5)
    ax[i].set_title(f'day {i+1}'); ax[i].axis('off')

## 13. Per-channel ablation — which channels does the model use?
Zero each dynamic channel, re-run, plot the change. Catches a wind-ignored regression.


In [ ]:
base = prob.copy(); deltas={}
with torch.no_grad():
    for c,name in enumerate(new_order):
        xd2 = xd.clone(); xd2[:,:,c]=0
        p2 = torch.sigmoid(model(xd2,xs))[0,0].cpu().numpy()
        deltas[name]=float(np.abs(p2-base).mean())
import pandas as pd; pd.Series(deltas).sort_values(ascending=False)

## 14. Save eval summary -> models/eval/<event>_<sha>.json


In [ ]:
import hashlib
sha=hashlib.sha256(Path(MODEL_PATH).read_bytes()).hexdigest()[:12] if Path(MODEL_PATH).exists() else 'nofile'
out=Path('models/eval'); out.mkdir(parents=True,exist_ok=True)
summary={'event':EVENT['name'],'ckpt_sha':sha,'steps':STEPS,
         'metrics':[{'day':i+1} for i in range(len(frames))]}
(out/f"{EVENT['name'].lower()}_{sha}.json").write_text(json.dumps(summary,indent=2))
print('wrote', out)